In [3]:
import numpy as np
import time
import sys

# ============================================================================
# 1. Создание массивов и их атрибуты
# ============================================================================

# Создаём массив из списка
a = np.array([1, 2, 3, 4, 5])
print("1. Базовый массив:", a)
print("   shape:", a.shape)          # (5,) – одномерный
print("   dtype:", a.dtype)          # int64 (зависит от платформы)
print("   itemsize:", a.itemsize, "байт")  # размер одного элемента
print("   nbytes:", a.nbytes, "байт")      # общий размер
print()

# Многомерный массив (матрица 3x4)
b = np.array([[1, 2, 3, 4],
              [5, 6, 7, 8],
              [9, 10, 11, 12]])
print("2. Матрица 3x4:\n", b)
print("   shape:", b.shape)           # (3, 4)
print("   strides:", b.strides)       # (32, 8) для float64? здесь int64
# strides: чтобы перейти на следующую строку – 4 элемента * itemsize
# чтобы перейти на следующий столбец – 1 элемент * itemsize
print()

# Специальные функции создания
zeros = np.zeros((2, 3))      # матрица из нулей
ones = np.ones((3, 2))        # матрица из единиц
eye = np.eye(4)               # единичная матрица 4x4
random = np.random.rand(3, 3) # случайные числа [0,1)
arange = np.arange(0, 10, 2)  # [0,2,4,6,8]
linspace = np.linspace(0, 1, 5) # [0, 0.25, 0.5, 0.75, 1]

print("3. Специальные массивы:")
print("   zeros:\n", zeros)
print("   ones:\n", ones)
print("   eye:\n", eye)
print("   arange:", arange)
print("   linspace:", linspace)
print()

# ============================================================================
# 2. Почему NumPy быстрее? Сравнение со списками
# ============================================================================

print("=" * 50)
print("4. Сравнение производительности NumPy и списков Python")
print("=" * 50)

size = 10_000_000
print(f"Размер: {size} элементов")

# Списки Python
list_a = list(range(size))
list_b = list(range(size))
start = time.time()
list_c = [list_a[i] + list_b[i] for i in range(size)]
list_time = time.time() - start
print(f"Списки Python (цикл): {list_time:.4f} сек")

# NumPy
np_a = np.arange(size)
np_b = np.arange(size)
start = time.time()
np_c = np_a + np_b
np_time = time.time() - start
print(f"NumPy (векторизованное сложение): {np_time:.4f} сек")
print(f"Ускорение: {list_time / np_time:.1f}x")
print()

# Матричное умножение (дополнительная демонстрация)
print("5. Умножение матриц: вложенные циклы vs NumPy")
N = 200
A = np.random.rand(N, N)
B = np.random.rand(N, N)

# Наивное умножение (циклы Python)
def matmul_python(A, B):
    n = A.shape[0]
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            s = 0.0
            for k in range(n):
                s += A[i, k] * B[k, j]
            C[i][j] = s
    return np.array(C)

start = time.time()
C_py = matmul_python(A, B)
time_py = time.time() - start
print(f"  Циклы Python: {time_py:.4f} сек")

start = time.time()
C_np = np.dot(A, B)          # или A @ B
time_np = time.time() - start
print(f"  NumPy (BLAS): {time_np:.4f} сек")
print(f"  Ускорение: {time_py / time_np:.1f}x")
print()

# ============================================================================
# 3. Порядок хранения в памяти (C-order vs Fortran-order)
# ============================================================================

print("=" * 50)
print("6. Порядок хранения в памяти (C-order vs Fortran-order)")
print("=" * 50)

# Создаём массив в C-порядке (по умолчанию)
c_arr = np.arange(12).reshape(3, 4)
print("Исходный массив (C-order):\n", c_arr)
print("strides:", c_arr.strides)   # (32, 8) для int64 – переход по строке 32 байта (4*8)

# Создаём массив в Fortran-порядке (столбцы идут подряд)
f_arr = np.asfortranarray(c_arr)   # или np.array(..., order='F')
print("Массив с Fortran-order:\n", f_arr)
print("strides:", f_arr.strides)   # (8, 24) – переход по столбцу 8 байт, по строке 24 байта (3*8)

# Транспонирование – меняется порядок, но данные не копируются
transposed = c_arr.T
print("Транспонированный массив (view):\n", transposed)
print("strides transposed:", transposed.strides)  # (8, 32) – шаги поменялись местами
print()

# ============================================================================
# 4. View vs Copy
# ============================================================================

print("=" * 50)
print("7. View (представление) vs Copy (копия)")
print("=" * 50)

arr = np.arange(10)
print("Исходный массив:", arr)

# Срез – view
slice_view = arr[2:5]
slice_view[0] = 100  # изменяем view
print("После изменения view (arr[2:5][0] = 100):", arr)  # исходный массив изменился
print("slice_view является view?", np.shares_memory(arr, slice_view))  # True

# Fancy indexing – копия
fancy_copy = arr[[2, 3, 4]]
fancy_copy[0] = 200
print("После изменения fancy_copy (копии):", arr)  # исходный не изменился
print("fancy_copy является view?", np.shares_memory(arr, fancy_copy))  # False

# Булева индексация – копия
mask = arr > 5
bool_copy = arr[mask]
bool_copy[0] = 999
print("После изменения bool_copy (копии):", arr)  # исходный не изменился
print()

# ============================================================================
# 5. Broadcasting (распространение)
# ============================================================================

print("=" * 50)
print("8. Broadcasting – операции с массивами разной формы")
print("=" * 50)

# Скаляр + массив
a = np.array([1, 2, 3])
print("a =", a)
print("a + 10 =", a + 10)  # скаляр "растягивается"

# Вектор-строка + вектор-столбец → матрица
row = np.array([1, 2, 3])
col = np.array([[4], [5], [6]])
print("\nВектор-строка:", row, "shape =", row.shape)
print("Вектор-столбец:\n", col, "shape =", col.shape)
result = row + col
print("Результат сложения (broadcasting):\n", result)
print("Форма результата:", result.shape)  # (3, 3)

# Пример с матрицей и вектором
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])
vector = np.array([10, 20, 30])
print("\nМатрица:\n", matrix)
print("Вектор:", vector)
print("Матрица + вектор (добавляется к каждой строке):\n", matrix + vector)
print()

# ============================================================================
# 6. Индексация и срезы
# ============================================================================

print("=" * 50)
print("9. Индексация и срезы")
print("=" * 50)

M = np.array([[ 0,  1,  2,  3],
              [ 4,  5,  6,  7],
              [ 8,  9, 10, 11],
              [12, 13, 14, 15]])
print("Матрица M:\n", M)

# Базовые срезы
print("M[1:3, 1:3] (подматрица 2x2):\n", M[1:3, 1:3])     # view
print("M[:, 2] (второй столбец):", M[:, 2])                # view
print("M[1, :] (вторая строка):", M[1, :])                 # view

# Fancy indexing (список индексов)
print("M[[0, 2], [1, 3]] (элементы (0,1) и (2,3)):", M[[0, 2], [1, 3]])  # копия

# Булева индексация
mask = M > 10
print("M > 10:\n", mask)
print("M[M > 10]:", M[M > 10])  # одномерный массив (копия)

# ============================================================================
# 7. Ravel vs Flatten
# ============================================================================

print("=" * 50)
print("10. Ravel (view) vs Flatten (копия)")
print("=" * 50)

arr2d = np.array([[1, 2, 3],
                  [4, 5, 6]])
print("Исходный массив:\n", arr2d)

# ravel – обычно view
raveled = arr2d.ravel()
print("arr2d.ravel():", raveled)
raveled[0] = 999
print("После изменения raveled[0] = 999, исходный массив:\n", arr2d)  # изменился

# flatten – всегда копия
flattened = arr2d.flatten()
print("arr2d.flatten():", flattened)
flattened[0] = 111
print("После изменения flattened[0] = 111, исходный массив:\n", arr2d)  # не изменился

print("ravel является view?", np.shares_memory(arr2d, raveled))   # True
print("flatten является view?", np.shares_memory(arr2d, flattened)) # False

# reshape(-1) – тоже view (если возможно)
reshaped = arr2d.reshape(-1)
print("arr2d.reshape(-1):", reshaped)
print("reshape является view?", np.shares_memory(arr2d, reshaped))  # True
print()

# ============================================================================
# 8. Дополнительные полезные операции
# ============================================================================

print("=" * 50)
print("11. Агрегация, универсальные функции, изменение формы")
print("=" * 50)

arr = np.arange(1, 10).reshape(3, 3)
print("Массив:\n", arr)
print("Сумма всех элементов:", np.sum(arr))
print("Сумма по столбцам (axis=0):", np.sum(arr, axis=0))
print("Сумма по строкам (axis=1):", np.sum(arr, axis=1))
print("Среднее:", np.mean(arr))
print("Стандартное отклонение:", np.std(arr))

# Универсальные функции (ufunc)
print("sqrt:\n", np.sqrt(arr))
print("exp:\n", np.exp(arr[:2, :2]))
print("sin:\n", np.sin(arr))

# Изменение формы
print("reshape(1,9):", arr.reshape(1, 9))
print("reshape(9,1):\n", arr.reshape(9, 1))
print("ravel():", arr.ravel())



1. Базовый массив: [1 2 3 4 5]
   shape: (5,)
   dtype: int64
   itemsize: 8 байт
   nbytes: 40 байт

2. Матрица 3x4:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
   shape: (3, 4)
   strides: (32, 8)

3. Специальные массивы:
   zeros:
 [[0. 0. 0.]
 [0. 0. 0.]]
   ones:
 [[1. 1.]
 [1. 1.]
 [1. 1.]]
   eye:
 [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]
   arange: [0 2 4 6 8]
   linspace: [0.   0.25 0.5  0.75 1.  ]

4. Сравнение производительности NumPy и списков Python
Размер: 10000000 элементов
Списки Python (цикл): 0.3787 сек
NumPy (векторизованное сложение): 0.0078 сек
Ускорение: 48.5x

5. Умножение матриц: вложенные циклы vs NumPy
  Циклы Python: 1.1395 сек
  NumPy (BLAS): 0.0002 сек
  Ускорение: 6004.5x

6. Порядок хранения в памяти (C-order vs Fortran-order)
Исходный массив (C-order):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
strides: (32, 8)
Массив с Fortran-order:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
strides: (8, 24)
Транспонированный массив (view